# Identity-panel experiments on a Kaggle GPU (open-weights model, no API)
Runs **all 3 tests / 8 conditions** on a model downloaded from Hugging Face, for the test cases in `CASES`. Every case is shown to the judges with its **full author profile** from `popquorn_biases.jsonl` (gender, race, age, occupation, education). Progress is kept in a **Hugging Face dataset**, so every Kaggle session resumes where the last one stopped.

**One-time setup** (full walkthrough: `SETUP_GUIDE.md`)
1. Right sidebar -> *Session options*: **Accelerator = GPU T4 x2** (a 7B model in fp16 needs ~15 GB, so it is split over two T4s; on a single 16 GB GPU use the 3B model or set `LOAD_IN_4BIT = True`). *Internet = On*.
2. *Add-ons -> Secrets*: add `HF_TOKEN` (Hugging Face token with **write** access) and tick *Attached*.
3. Edit the next cell (`MODEL` and `CASES` are the ones you were assigned), then **Save Version -> Save & Run All (Commit)** so it keeps running after you close the tab.

The session stops itself cleanly after `MAX_MINUTES` and uploads everything. To continue, run the notebook again **with the same MODEL and CASES**: it picks up at the first unfinished case. Results are named after the model and range (e.g. `results_7b_cases_0-2000.xlsx`), so teammates never overwrite each other.

In [ ]:
HF_REPO      = "YOUR_HF_USERNAME/identity-panel-results"   # private dataset repo; created automatically
CODE_URL     = "https://github.com/YOUR_GITHUB_USERNAME/aditi-dsa.git"
MODEL        = "Qwen/Qwen2.5-7B-Instruct"      # 7B model; the 3B model is "Qwen/Qwen2.5-3B-Instruct"
CASES        = "0-2000"     # your assigned test cases: rows of popquorn.jsonl, 0 = first row, both ends included
LOAD_IN_4BIT = False        # True = less GPU memory (needs bitsandbytes), usually slower
ROUNDS       = 2            # discussion rounds in the discussion conditions
MAX_MINUTES  = 600          # stop cleanly after this long (Kaggle sessions are capped around 12 h)
VERIFY_FIRST = True         # first check that the biased judge really behaves differently (a few minutes)

In [ ]:
import os, subprocess, sys
CODE_DIR = "/kaggle/working/aditi-dsa"        # or the path of a Kaggle Dataset holding the code
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", CODE_URL, CODE_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate",
                "huggingface_hub", "bitsandbytes", "openpyxl"], check=True)
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - set Accelerator to GPU!")

In [ ]:
# The model is downloaded here (and cached for this session). HF_TOKEN is only needed for gated models.
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

common = ["--items", f"{CODE_DIR}/examples/popquorn.jsonl", "--biases", f"{CODE_DIR}/examples/popquorn_biases.jsonl",
          "--model", MODEL, "--cases", CASES, "--rounds", str(ROUNDS), "--workers", "32", "--max-batch", "16"]
if LOAD_IN_4BIT:
    common.append("--load-in-4bit")

if VERIFY_FIRST:
    rc = subprocess.run([sys.executable, "-u", f"{CODE_DIR}/scripts/run_suite.py", *common,
                         "--verify-bias", "--verify-n", "8", "--out-dir", "/kaggle/working/verify"]).returncode
    assert rc == 0, ("The biased judge did not behave differently from a normal one with this model - "
                     "see the output above. Try a stronger model, or set VERIFY_FIRST = False to run anyway.")

In [ ]:
args = ["--hf-repo", HF_REPO, "--max-runtime-minutes", str(MAX_MINUTES), "--", *common]
rc = subprocess.run([sys.executable, "-u", f"{CODE_DIR}/scripts/kaggle_runner.py", *args]).returncode
print("exit code:", rc, "(0 = done, or stopped cleanly at the time cap -> run the notebook again to continue)")

Results are in your Hugging Face dataset (Files tab), named after the model and case range, e.g. for 7B cases 0-2000:
- `results_7b_cases_0-2000.xlsx` - Excel workbook: `runs` (one row per case x condition, with `model`/`model_size` and every author label), `judge_evaluations` (every judge's score **and reasoning** in every round), `summary`.
- `results_7b_cases_0-2000.jsonl` - the raw results; **this is the file the analysis uses**.
- `results_7b_cases_0-2000.cache.jsonl` - needed to resume; keep it on the Hub.
- `results_7b_cases_0-2000_summary.csv` - mean score and gender gap per condition.